In [ ]:
import os
import re
import csv
import json
import random
from pathlib import Path
from typing import List, Dict, Any, Tuple

import torch
import numpy as np
from PIL import Image, ImageDraw, ImageFont

import matplotlib.pyplot as plt
import matplotlib.patches as patches

from huggingface_hub import login
from transformers import (
    AutoProcessor,
    AutoModelForZeroShotObjectDetection,
    # Sam2Model
)
# import open_clip
# from ultralytics import YOLO

# from google.colab import userdata
from tqdm.notebook import tqdm

In [ ]:
hf_token = userdata.get('HF_TOKEN')
login(token=hf_token, add_to_git_credential=False)

In [27]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
IMAGE_PATH = "tree.jpg"

GROUNDING_DINO_ID = "IDEA-Research/grounding-dino-tiny"
SAM2_ID = "facebook/sam2-hiera-base-plus"
BIOCLIP_ID = "hf-hub:imageomics/bioclip"

In [28]:
def load_all_models() -> Dict[str, Any]:
    models = {}
    print("1/3: Grounding DINO...")
    models['dino_processor'] = AutoProcessor.from_pretrained(GROUNDING_DINO_ID)
    models['dino_model'] = AutoModelForZeroShotObjectDetection.from_pretrained(GROUNDING_DINO_ID).to(DEVICE)
    models['dino_model'].eval()

    # print("2/3: SAM2...")
    # models['sam_processor'] = AutoProcessor.from_pretrained(SAM2_ID)
    # models['sam_model'] = Sam2Model.from_pretrained(SAM2_ID).to(DEVICE)
    # models['sam_model'].eval()

    # print("3/3: BioCLIP...")
    # model, _, preprocess = open_clip.create_model_and_transforms(BIOCLIP_ID)
    # models['bioclip_model'] = model.to(DEVICE)
    # models['bioclip_preprocess'] = preprocess
    # models['bioclip_tokenizer'] = open_clip.get_tokenizer(BIOCLIP_ID)
    # models['bioclip_model'].eval()
    
    return models

In [33]:
models = load_all_models()

1/3: Grounding DINO...


Could not load the custom kernel for multi-scale deformable attention: DLL load failed while importing MultiScaleDeformableAttention: Не найден указанный модуль.
Could not load the custom kernel for multi-scale deformable attention: DLL load failed while importing MultiScaleDeformableAttention: Не найден указанный модуль.
Could not load the custom kernel for multi-scale deformable attention: DLL load failed while importing MultiScaleDeformableAttention: Не найден указанный модуль.
Could not load the custom kernel for multi-scale deformable attention: DLL load failed while importing MultiScaleDeformableAttention: Не найден указанный модуль.
Could not load the custom kernel for multi-scale deformable attention: DLL load failed while importing MultiScaleDeformableAttention: Не найден указанный модуль.
Could not load the custom kernel for multi-scale deformable attention: DLL load failed while importing MultiScaleDeformableAttention: Не найден указанный модуль.
Could not load the custom ke

In [30]:
def detect_objects(
    dino_model,
    dino_processor,
    image: Image.Image
) -> Dict[str, Any]:
    text_labels = [["a tree", "a shrub"]]
    inputs = dino_processor(images=image, text=text_labels, return_tensors="pt").to(DEVICE)
    
    with torch.no_grad():
        outputs = dino_model(**inputs)
        
    results = dino_processor.post_process_grounded_object_detection(
        outputs,
        threshold=0.40,
        text_threshold=0.25,
        target_sizes=[image.size[::-1]]
    )
    res = results[0]

    boxes = res.get("boxes", torch.empty((0,4))).detach().cpu() if torch.is_tensor(res.get("boxes", None)) else torch.tensor(res.get("boxes", []), dtype=torch.float32)
    scores = res.get("scores", torch.empty((0,))).detach().cpu() if torch.is_tensor(res.get("scores", None)) else torch.tensor(res.get("scores", []), dtype=torch.float32)
    text_labels = res.get("text_labels", [])

    if boxes.numel() == 0:
        boxes = torch.empty((0,4), dtype=torch.float32)
    if scores.numel() == 0:
        scores = torch.empty((0,), dtype=torch.float32)

    return {"boxes": boxes.cpu(), "scores": scores.cpu(), "text_labels": list(text_labels)}

In [42]:
def detect_multiple_objects(
    dino_model,
    dino_processor,
    image: Image.Image,
    target_classes: List[str] = ["tree", "shrub"],
    max_objects_per_class: int = 5
) -> Dict[str, Any]:
    all_boxes = []
    all_scores = []
    all_labels = []

    for class_name in target_classes:
        text_labels = [[f"a {class_name}"]]
        
        inputs = dino_processor(images=image, text=text_labels, return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            outputs = dino_model(**inputs)
            
        results = dino_processor.post_process_grounded_object_detection(
            outputs,
            threshold=0.40,
            text_threshold=0.25,
            target_sizes=[image.size[::-1]]
        )[0]

        num_found = len(results['boxes'])
        if num_found > 0:
            num_to_keep = min(num_found, max_objects_per_class)
            
            all_boxes.append(results['boxes'][:num_to_keep])
            all_scores.append(results['scores'][:num_to_keep])
            all_labels.extend([class_name] * num_to_keep)
        
    if not all_boxes:
        return {
            "boxes": torch.empty((0, 4), device=DEVICE),
            "scores": torch.empty(0, device=DEVICE),
            "text_labels": []
        }

    final_boxes = torch.cat(all_boxes, dim=0)
    final_scores = torch.cat(all_scores, dim=0)
    
    merged_results = {
        "scores": final_scores,
        "boxes": final_boxes,
        "text_labels": all_labels
    }
    
    return merged_results

In [34]:
INPUT_DIR = Path("data/Санитарка/")
OUTPUT_DIR = Path("data/crops_dataset")
CROPS_SUBDIR = OUTPUT_DIR / "crops"
MAX_PER_CLASS = 5
TARGET_CLASSES = ["tree", "shrub"]
SCORE_FMT = "{:.3f}"

In [40]:
CROPS_SUBDIR.mkdir(parents=True, exist_ok=True)

In [36]:
def _sanitize_filename(s: str) -> str:
    s = str(s)
    s = re.sub(r"[^\w\-_\. ]", "_", s)
    return s

In [37]:
def _to_numpy_boxes_scores(res: dict):
    boxes = res.get("boxes", None)
    scores = res.get("scores", None)
    labels = res.get("text_labels", []) or []

    if boxes is None:
        boxes_np = np.zeros((0,4), dtype=np.float32)
    elif torch.is_tensor(boxes):
        boxes_np = boxes.detach().cpu().numpy()
    else:
        boxes_np = np.array(boxes, dtype=np.float32)

    if scores is None:
        scores_np = np.zeros((boxes_np.shape[0],), dtype=np.float32)
    elif torch.is_tensor(scores):
        scores_np = scores.detach().cpu().numpy()
    else:
        scores_np = np.array(scores, dtype=np.float32)

    # pad labels if shorter
    if len(labels) < boxes_np.shape[0]:
        labels = labels + [""] * (boxes_np.shape[0] - len(labels))

    return boxes_np, scores_np, list(labels)

In [38]:
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tiff", ".webp"}

In [43]:
files = [p for p in sorted(INPUT_DIR.iterdir()) if p.suffix.lower() in IMG_EXTS]

manifest_rows = []
row_header = ["orig_path","crop_path","obj_index","class","score","x1","y1","x2","y2","orig_width","orig_height"]

for img_path in tqdm(files, desc="Processing images"):
    try:
        img = Image.open(img_path).convert("RGB")
    except Exception as e:
        print(f"Failed to open {img_path}: {e}")
        continue

    W, H = img.size

    try:
        # res = detect_multiple_objects(models['dino_model'], models['dino_processor'],
        #                               img, target_classes=TARGET_CLASSES, max_objects_per_class=MAX_PER_CLASS)
        res = detect_objects(models['dino_model'], models['dino_processor'], img)
    except Exception as e:
        print(f"Detection failed for {img_path}: {e}")
        continue

    boxes_np, scores_np, labels = _to_numpy_boxes_scores(res)

    if boxes_np.shape[0] == 0:
        continue

    for i in range(boxes_np.shape[0]):
        x1, y1, x2, y2 = boxes_np[i].astype(int).tolist()
        # clamp coordinates to image bounds
        x1 = max(0, min(x1, W-1))
        x2 = max(0, min(x2, W))
        y1 = max(0, min(y1, H-1))
        y2 = max(0, min(y2, H))

        # skip invalid boxes
        if x2 <= x1 or y2 <= y1:
            continue

        crop = img.crop((x1, y1, x2, y2))

        label = labels[i] if i < len(labels) else ""
        score = float(scores_np[i]) if i < len(scores_np) else 0.0

        # сформируем понятное имя файла
        base = img_path.stem
        label_safe = _sanitize_filename(label or "obj")
        crop_name = f"{base}_obj{i+1}_{label_safe}_{SCORE_FMT.format(score)}.jpg"
        crop_path = CROPS_SUBDIR / crop_name

        try:
            crop.save(crop_path, quality=95)
        except Exception as e:
            # fallback to png if jpeg fails
            crop_path = CROPS_SUBDIR / (crop_name.replace(".jpg", ".png"))
            crop.save(crop_path)

        manifest_rows.append({
            "orig_path": str(img_path),
            "crop_path": str(crop_path),
            "obj_index": i+1,
            "class": label,
            "score": score,
            "x1": int(x1),
            "y1": int(y1),
            "x2": int(x2),
            "y2": int(y2),
            "orig_width": int(W),
            "orig_height": int(H)
        })

Processing images:  67%|██████▋   | 704/1054 [06:43<03:34,  1.63it/s]

Failed to open data\Санитарка\IMG_20210917_125525.jpg: image file is truncated (27 bytes not processed)


Processing images: 100%|██████████| 1054/1054 [10:20<00:00,  1.70it/s]


In [44]:
import csv

manifest_file = OUTPUT_DIR / "manifest.csv"
with open(manifest_file, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=row_header)
    writer.writeheader()
    for r in manifest_rows:
        writer.writerow(r)

print(f"Done. Saved {len(manifest_rows)} crops to {CROPS_SUBDIR} and manifest to {manifest_file}")

Done. Saved 751 crops to data\crops_dataset\crops and manifest to data\crops_dataset\manifest.csv
